In [38]:
import pandas as pd
import numpy as np
import ast

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

In [39]:
ptb_df = pd.read_csv("/Users/tejas/Documents/C_ML/Heart_disease_CML/data/ptb_xl/ptbxl_database.csv")
scp_df = pd.read_csv("/Users/tejas/Documents/C_ML/Heart_disease_CML/data/ptb_xl/scp_statements.csv")

print(ptb_df.shape)
print(scp_df.shape)

(21799, 28)
(71, 13)


In [40]:
scp_df = scp_df[scp_df["diagnostic"] == 1]
scp_to_superclass = (
    scp_df.set_index("Unnamed: 0")["diagnostic_class"]
    .to_dict()
)
list(scp_to_superclass.items())[:10]


[('NDT', 'STTC'),
 ('NST_', 'STTC'),
 ('DIG', 'STTC'),
 ('LNGQT', 'STTC'),
 ('NORM', 'NORM'),
 ('IMI', 'MI'),
 ('ASMI', 'MI'),
 ('LVH', 'HYP'),
 ('LAFB', 'CD'),
 ('ISC_', 'STTC')]

In [41]:

type(ptb_df["scp_codes"].iloc[0])

str

In [42]:
def get_superclass(scp_dict):
    for code in scp_dict.keys():
        if code in scp_to_superclass:
            return scp_to_superclass[code]
    return "OTHER"

ptb_df["diagnostic_superclass"] = ptb_df["scp_codes"].apply(get_superclass)
ptb_df["diagnostic_superclass"].value_counts()

AttributeError: 'str' object has no attribute 'keys'

In [ ]:
def map_to_3class(label):
    if label == "NORM":
        return "NORMAL"
    elif label == "MI":
        return "MI"
    else:
        return "OTHER_ABN"

ptb_df["target"] = ptb_df["diagnostic_superclass"].apply(map_to_3class)
ptb_df["target"].value_counts()

target
NORMAL       9514
OTHER_ABN    6861
MI           5424
Name: count, dtype: int64

In [ ]:
features = [
    "age",
    "sex",
    "pacemaker",
    "strat_fold"
]
df = ptb_df[features + ["target"]].copy()
df["age"] = df["age"].fillna(df["age"].median())
df["sex"] = df["sex"].fillna(0)
df["pacemaker"] = df["pacemaker"].fillna(0)
df["pacemaker"] = df["pacemaker"].apply(lambda x: 1 if x == "ja, pacemaker" else 0)

In [ ]:
train_df = df[df["strat_fold"] <= 8]
test_df = df[df["strat_fold"] > 8]
X_train = train_df.drop(columns=["target", "strat_fold"])
X_test = test_df.drop(columns=["target", "strat_fold"])

y_train = train_df["target"]
y_test = test_df["target"]


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

print("Classes:", le.classes_)

Classes: ['MI' 'NORMAL' 'OTHER_ABN']


In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_model.fit(X_train, y_train_enc)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

In [ ]:
from sklearn.metrics import classification_report

y_pred = xgb_model.predict(X_test)

print(classification_report(y_test_enc, y_pred, target_names=le.classes_))

              precision    recall  f1-score   support

          MI       0.32      0.23      0.27      1079
      NORMAL       0.61      0.71      0.66      1918
   OTHER_ABN       0.43      0.42      0.42      1384

    accuracy                           0.50      4381
   macro avg       0.45      0.45      0.45      4381
weighted avg       0.48      0.50      0.49      4381



In [45]:
import joblib
joblib.dump(xgb_model, "/Users/tejas/Documents/C_ML/Heart_disease_CML/Jupyter_NB/saved_models/ptb_xgb_model.pkl")
joblib.dump(le, "/Users/tejas/Documents/C_ML/Heart_disease_CML/Jupyter_NB/saved_models/ptb_label_encoder.pkl")

print("PTB binary MI model saved.")

PTB binary MI model saved.
